In [ ]:
# CONFIGURATION

CLEAN_PATH      = "cleaned_interactions.jsonl"
MIN_REVIEWS     = 3       # minimum interactions per user to be included
FETCH_TIMEOUT   = 30      # seconds to wait for each HTTP response

BASE_URL = (
    "https://huggingface.co/datasets/McAuley-Lab/"
    "Amazon-Reviews-2023/resolve/main/raw/review_categories/{}.jsonl"
)

# ── MULTI-DATASET CHANGE ── (1 of 2) 
# CURRENT  : single category for All_Beauty only.
# TO EXPAND: replace the list below with the full category list.
#            The full list is provided as a comment block directly below
#            so you can copy-paste it when ready — no other change needed here.
#
# FULL LIST (uncomment and replace CATEGORIES when switching to all datasets):
#
# CATEGORIES = [
#     "All_Beauty",
#     "Amazon_Fashion",
#     "Appliances",
#     "Arts_Crafts_and_Sewing",
#     "Automotive",
#     "Baby_Products",
#     "Beauty_and_Personal_Care",
#     "Books",
#     "CDs_and_Vinyl",
#     "Cell_Phones_and_Accessories",
#     "Clothing_Shoes_and_Jewelry",
#     "Digital_Music",
#     "Electronics",
#     "Gift_Cards",
#     "Grocery_and_Gourmet_Food",
#     "Handmade_Products",
#     "Health_and_Household",
#     "Home_and_Kitchen",
#     "Industrial_and_Scientific",
#     "Kindle_Store",
#     "Magazine_Subscriptions",
#     "Movies_and_TV",
#     "Musical_Instruments",
#     "Office_Products",
#     "Patio_Lawn_and_Garden",
#     "Pet_Supplies",
#     "Software",
#     "Sports_and_Outdoors",
#     "Subscription_Boxes",
#     "Tools_and_Home_Improvement",
#     "Toys_and_Games",
#     "Video_Games",
#     "Unknown",
# ]

CATEGORIES = [
    "All_Beauty",    #  only this category active right now
]
# ── END MULTI-DATASET CHANGE (1 of 2) 

print(f"Categories to process : {CATEGORIES}")
print(f"Output path           : {CLEAN_PATH}")
print(f"Min reviews per user  : {MIN_REVIEWS}")

Categories to process : ['All_Beauty']
Output path           : cleaned_interactions.jsonl
Min reviews per user  : 3


In [ ]:
import json
import requests
# STREAMING HELPER

def stream_category(category: str):
    """
    Stream one category JSONL file from HuggingFace line by line.
    Yields parsed record dicts.  Skips empty or malformed lines silently.
    Attaches a 'category' field to every record for downstream filtering.
    """
    url = BASE_URL.format(category)
    print(f"  Connecting to: {url}")
    with requests.get(url, stream=True, timeout=FETCH_TIMEOUT) as resp:
        resp.raise_for_status()
        for raw_line in resp.iter_lines():
            if not raw_line:
                continue
            try:
                record = json.loads(raw_line)
                record["category"] = category
                yield record
            except json.JSONDecodeError:
                continue

print("stream_category() defined.")

stream_category() defined.


In [ ]:
# VALIDATION HELPERS - shared logic used for every category

def validate_rating(rating) -> float | None:
    """Return float in [1.0, 5.0] or None if invalid / out of range."""
    try:
        r = float(rating)
        return r if 1.0 <= r <= 5.0 else None
    except (TypeError, ValueError):
        return None


def has_valid_image(images) -> bool:
    """
    Return True if the images list contains at least one non-null HTTP URL.
    Handles both list-of-dicts  (review images)
    and   plain string URLs     (some older records).
    """
    if not images:
        return False
    priority = ["hi_res", "large_image_url", "large",
                "medium_image_url", "small_image_url", "thumb"]
    for img in images:
        if isinstance(img, dict):
            for key in priority:
                url = img.get(key)
                if url and isinstance(url, str) and url.startswith("http"):
                    return True
        elif isinstance(img, str) and img.startswith("http"):
            return True
    return False


def has_valid_text(text, title="") -> bool:
    """Return True if text or title contains any non-whitespace content."""
    combined = (str(text or "") + str(title or "")).strip()
    return len(combined) > 0


def is_valid_record(r: dict) -> bool:
    """
    Hard requirements every record must satisfy to be kept:
      1. user_id and asin must be present and non-empty.
      2. rating must be a float in [1.0, 5.0].
      3. At least one modality (text OR image) must be usable.
         Records with only one modality are still kept — the embedding
         notebooks handle the case where their modality is absent.
         Only the merged file used for prototype training requires both.
    """
    if not r.get("user_id") or not r.get("asin"):
        return False
    if validate_rating(r.get("rating")) is None:
        return False
    if not has_valid_text(r.get("text"), r.get("title"))        and not has_valid_image(r.get("images", [])):
        return False
    return True


print("Validation helpers defined:")
print("  validate_rating(rating)        → float | None")
print("  has_valid_image(images)        → bool")
print("  has_valid_text(text, title)    → bool")
print("  is_valid_record(record)        → bool")

Validation helpers defined:
  validate_rating(rating)        → float | None
  has_valid_image(images)        → bool
  has_valid_text(text, title)    → bool
  is_valid_record(record)        → bool


In [ ]:
from collections import defaultdict

# IN-MEMORY SECTION (unnecessary switching to all datasets)

class InMemoryDedup:
    """
    Deduplicates (user_id, asin) pairs in memory.
    For each key, keeps the record with the latest timestamp.
    """
    def __init__(self):
        self._store  = {}   # (user_id, asin) → record dict
        self._counts = defaultdict(int)   # user_id → interaction count

    def upsert(self, record: dict):
        """Insert record, or replace existing if this timestamp is newer."""
        key = (record["user_id"], record["asin"])
        ts  = record.get("timestamp") or 0
        if key not in self._store:
            self._store[key]  = record
            self._counts[record["user_id"]] += 1
        elif ts > (self._store[key].get("timestamp") or 0):
            self._store[key] = record
            # count does not change — same user, same asin

    def qualified_records(self, min_reviews: int):
        """Yield records belonging to users with >= min_reviews interactions."""
        qualified = {uid for uid, cnt in self._counts.items()
                     if cnt >= min_reviews}
        for record in self._store.values():
            if record["user_id"] in qualified:
                yield record

    def stats(self) -> dict:
        return {
            "unique_interactions": len(self._store),
            "unique_users"       : len(self._counts),
        }

dedup = InMemoryDedup()
print("Using InMemoryDedup store.")

# SQLITE SECTION (iwhen switching to all datasets)

# import sqlite3
#
# class SQLiteDedup:
#     """
#     Deduplicates (user_id, asin) pairs using SQLite.
#     Keeps the record with the latest timestamp per key.
#     Use this when the full dataset exceeds available RAM.
#     """
#     def __init__(self, db_path: str = "dedup_cache.db"):
#         self.conn = sqlite3.connect(db_path)
#         self.conn.execute("""
#             CREATE TABLE IF NOT EXISTS interactions (
#                 user_id  TEXT,
#                 asin     TEXT,
#                 ts       INTEGER,
#                 record   TEXT,
#                 PRIMARY KEY (user_id, asin)
#             )
#         "")
#         self.conn.execute(
#             "CREATE INDEX IF NOT EXISTS idx_user ON interactions(user_id)"
#         )
#         self.conn.commit()
#
#     def upsert(self, record: dict):
#         ts = record.get("timestamp") or 0
#         self.conn.execute("""
#             INSERT INTO interactions(user_id, asin, ts, record)
#             VALUES (?, ?, ?, ?)
#             ON CONFLICT(user_id, asin) DO UPDATE SET
#                 ts     = excluded.ts,
#                 record = excluded.record
#             WHERE excluded.ts > interactions.ts
#         """, (record["user_id"], record["asin"], ts, json.dumps(record)))
#
#     def qualified_records(self, min_reviews: int):
#         self.conn.commit()
#         qualified_users = {
#             row[0] for row in self.conn.execute("""
#                 SELECT user_id FROM interactions
#                 GROUP BY user_id HAVING COUNT(*) >= ?
#             """, (min_reviews,))
#         }
#         for (record_json,) in self.conn.execute(
#             "SELECT record FROM interactions WHERE user_id IN ({})".format(
#                 ",".join("?" * len(qualified_users))
#             ), list(qualified_users)
#         ):
#             yield json.loads(record_json)
#
#     def stats(self) -> dict:
#         self.conn.commit()
#         n_interactions = self.conn.execute(
#             "SELECT COUNT(*) FROM interactions").fetchone()[0]
#         n_users = self.conn.execute(
#             "SELECT COUNT(DISTINCT user_id) FROM interactions").fetchone()[0]
#         return {
#             "unique_interactions": n_interactions,
#             "unique_users"       : n_users,
#         }
#
# dedup = SQLiteDedup(db_path="dedup_cache.db")
# print("Using SQLiteDedup store.")


Using InMemoryDedup store.


In [ ]:
from tqdm.auto import tqdm

# MAIN PROCESSING LOOP

stats = {
    "total_streamed" : 0,
    "invalid_dropped": 0,
}

for category in tqdm(CATEGORIES, desc="Categories"):
    print(f"\nProcessing: {category}")
    cat_streamed = 0
    cat_dropped  = 0

    for record in stream_category(category):
        stats["total_streamed"] += 1
        cat_streamed += 1

        # Normalise rating to float before storing
        record["rating"] = validate_rating(record.get("rating"))

        if not is_valid_record(record):
            stats["invalid_dropped"] += 1
            cat_dropped += 1
            continue

        dedup.upsert(record)

    print(f"  Streamed : {cat_streamed:>8,}")
    print(f"  Dropped  : {cat_dropped:>8,}")

dedup_stats = dedup.stats()
print(f"\n{'='*55}")
print(f"  Total streamed           : {stats['total_streamed']:>10,}")
print(f"  Invalid / dropped        : {stats['invalid_dropped']:>10,}")
print(f"  Unique (user,asin) pairs : {dedup_stats['unique_interactions']:>10,}")
print(f"  Unique users             : {dedup_stats['unique_users']:>10,}")
print(f"{'='*55}")

Categories:   0%|          | 0/1 [00:00<?, ?it/s]


Processing: All_Beauty
  Connecting to: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/All_Beauty.jsonl
  Streamed :  701,528
  Dropped  :        0

  Total streamed           :    701,528
  Invalid / dropped        :          0
  Unique (user,asin) pairs :    694,252
  Unique users             :    631,986


In [ ]:
# WRITE CLEANED OUTPUT

written = 0

with open(CLEAN_PATH, "w") as out_f:
    for record in tqdm(
        dedup.qualified_records(MIN_REVIEWS),
        desc="Writing cleaned records"
    ):
        out_f.write(json.dumps(record) + "\n")
        written += 1

print(f"\nRecords written → {CLEAN_PATH} : {written:,}")
print(f"These belong to users with >= {MIN_REVIEWS} interactions.")
print(f"\nNext steps:")
print(f"  1. Run all_beauty_BERT_embedding_updated.ipynb")
print(f"  2. Run all_beauty_ViT_embedding_updated.ipynb")
print(f"  3. Run merge.py")

Writing cleaned records: 0it [00:00, ?it/s]


Records written → cleaned_interactions.jsonl : 35,624
These belong to users with >= 3 interactions.

Next steps:
  1. Run all_beauty_BERT_embedding_updated.ipynb
  2. Run all_beauty_ViT_embedding_updated.ipynb
  3. Run merge.py


In [ ]:
# VERIFICATION — spot-check the output file
from collections import Counter

sample_records = []
with open(CLEAN_PATH) as f:
    for i, line in enumerate(f):
        if i >= 5000:   # read first 5000 lines for stats
            break
        sample_records.append(json.loads(line.strip()))

ratings_seen    = [r["rating"] for r in sample_records if r.get("rating")]
categories_seen = Counter(r.get("category", "Unknown") for r in sample_records)
has_text_count  = sum(1 for r in sample_records if has_valid_text(r.get("text"), r.get("title")))
has_image_count = sum(1 for r in sample_records if has_valid_image(r.get("images", [])))

print(f"Spot-check on first {len(sample_records):,} records of {CLEAN_PATH}:")
print(f"  Records with valid text  : {has_text_count:,}  ({100*has_text_count/len(sample_records):.1f}%)")
print(f"  Records with valid image : {has_image_count:,}  ({100*has_image_count/len(sample_records):.1f}%)")
print(f"  Rating range             : {min(ratings_seen):.1f} – {max(ratings_seen):.1f}")
print(f"  Mean rating              : {sum(ratings_seen)/len(ratings_seen):.3f}")
print()
print("  Categories in sample:")
for cat, cnt in categories_seen.most_common():
    print(f"    {cat:<35} : {cnt:>6,}")
print()
print("  Example record:")
ex = sample_records[0]
print(f"    user_id  : {ex['user_id']}")
print(f"    asin     : {ex['asin']}")
print(f"    category : {ex.get('category')}")
print(f"    rating   : {ex['rating']}")
print(f"    has text : {has_valid_text(ex.get('text'), ex.get('title'))}")
print(f"    has image: {has_valid_image(ex.get('images', []))}")

Spot-check on first 5,000 records of cleaned_interactions.jsonl:
  Records with valid text  : 5,000  (100.0%)
  Records with valid image : 782  (15.6%)
  Rating range             : 1.0 – 5.0
  Mean rating              : 4.211

  Categories in sample:
    All_Beauty                          :  5,000

  Example record:
    user_id  : AFSKPY37N3C43SOI5IEXEK5JSIYA
    asin     : B08P2DZB4X
    category : All_Beauty
    rating   : 5.0
    has text : True
    has image: False
